# TRUSTSR Lab 00 — Package Quickstart

**Status:** Synthetic demonstration of the `trustsr` package (v0.2.0) | Version 1.0.0

Unlike Labs 01-13, which are self-contained prototypes, this notebook **imports the tested package** (`src/trustsr`). It walks through the full fail-closed flow: versioned signed checkpoint, replay protection, audited recovery, five-gate verification, contamination classification, trust promotion and idempotent retries.

Install first from the repository root: `python -m pip install -e .`

**Evidence boundary:** all data is synthetic and keys are ephemeral. Passing assertions here shows code paths behave as designed; it is not evidence of real-world effectiveness.

In [1]:
import trustsr
from trustsr.anchor import HeadAnchor
from trustsr.audit import AuditLog
from trustsr.checkpoint import generate_keypair, sign_checkpoint
from trustsr.recovery import recover_state, recover_state_attempt
from trustsr.state_store import SqliteStateStore
from trustsr.replay import HighWaterMark, ReplayError
from trustsr.verification import verify_recovery
from trustsr.graph import classify_contamination, quarantine_set
from trustsr.trust_machine import TrustTracker, TrustLabel
from trustsr.idempotent import IdempotentReceiver
print("trustsr", trustsr.__version__)

trustsr 0.3.0


## 1. Versioned, signed checkpoints and a replay guard

In [2]:
private_key, public_key = generate_keypair()   # demo only: real keys need independent provisioning
cp_v1 = sign_checkpoint({"mode": "old-vulnerable", "counter": 1}, private_key, version=1)
cp_v2 = sign_checkpoint({"mode": "safe", "counter": 3}, private_key, version=2)

guard = HighWaterMark()
assert guard.register(1) and guard.register(2)
assert not guard.register(2)          # same version cannot be registered twice
print("High-water mark:", guard.mark)

High-water mark: 2


## 2. Audited full recovery and five-gate verification

The `trusted_audit_head` comes from a `HeadAnchor` written earlier, **not** from `log.head` of the log under test (that would be circular: a rewritten log always matches its own head).

In [3]:
import os, tempfile

manifest = {"expected_record_ids": ["e1", "e2"], "expected_count": 2}
log = AuditLog()
log.append("checkpoint_created", {"id": "cp-2"})
log.declare_scope(manifest)

contaminated = {"mode": "unsafe", "counter": 999, "backdoor": True}
work = tempfile.mkdtemp()
store = SqliteStateStore(os.path.join(work, "agent_state.db"))   # the agent's persistent state
store.write_state(contaminated)
attempt = recover_state_attempt(cp_v2, public_key, store.read_committed().state, audit_log=log, replay_guard=guard)
restored = store.commit_recovery(attempt).state                   # atomic commit + independent re-read
assert restored == {"mode": "safe", "counter": 3}            # injected key is dropped

# The expected audit head is recorded OUT-OF-BAND by the recovery controller and loaded later by the
# verifier. (Demo: a temp file. Real use: a separate trust domain the attacker cannot write.)
anchor = HeadAnchor(os.path.join(work, "audit_head.json"))
anchor.advance(log)

report = verify_recovery(checkpoint=cp_v2, public_key=public_key, audit_log=log, recovered_state=restored,
                         scope_manifest=manifest, observed_record_ids=["e1", "e2"],
                         recovery_id=attempt.recovery_id, trusted_audit_head=anchor.load(), require_anchored_head=True,
                         replay_guard=guard)
print(report["decision"], report["gates"])
assert report["decision"] == "PASS" and report["audit_head_anchored"] is True

PASS {'checkpoint_valid': True, 'audit_valid': True, 'recovery_complete': True, 'contamination_removed': True, 'scope_complete': True}


## 3. Replay attack: an old but validly signed checkpoint is refused

In [4]:
try:
    recover_state(cp_v1, public_key, restored, replay_guard=guard)
    raise AssertionError("replay should have been refused")
except ReplayError as exc:
    print("Refused:", exc)

Refused: Checkpoint version 1 is older than high-water mark 2


## 4. A contaminated result cannot pass

In [5]:
bad = verify_recovery(checkpoint=cp_v2, public_key=public_key, audit_log=log, recovered_state=contaminated,
                      scope_manifest=manifest, observed_record_ids=["e1", "e2"], replay_guard=guard)
print(bad["decision"], bad["contamination_report"]["changed_keys"], bad["contamination_report"]["extra_keys"])
assert bad["decision"] == "FAIL" and bad["gates"]["contamination_removed"] is False

FAIL ['counter', 'mode'] ['backdoor']


### 4b. A wholesale-rewritten audit log is caught only because the head was anchored


In [6]:
forged = AuditLog()
forged.append("checkpoint_created", {"id": "attacker-edited"})          # doctored first event
for event in log.events[1:]:
    forged.append(event.event_type, event.details)
assert forged.verify()                                                    # internally consistent
caught = verify_recovery(checkpoint=cp_v2, public_key=public_key, audit_log=forged, recovered_state=restored,
                         scope_manifest=manifest, observed_record_ids=["e1", "e2"],
                         trusted_audit_head=anchor.load(), replay_guard=guard)
blind = verify_recovery(checkpoint=cp_v2, public_key=public_key, audit_log=forged, recovered_state=restored,
                        scope_manifest=manifest, observed_record_ids=["e1", "e2"],
                        trusted_audit_head=forged.head, replay_guard=guard)   # circular: do NOT do this
print("anchored head ->", caught["decision"], "| circular head ->", blind["decision"])
assert caught["gates"]["audit_valid"] is False and caught["decision"] == "FAIL"
assert blind["gates"]["audit_valid"] is True      # the circular pattern is blind to the rewrite

anchored head -> FAIL | circular head -> PASS


## 5. Trust promotion only through a genuine verification result

`promote_after_verification` accepts only the sealed `VerificationReport` that `verify_recovery` returned. A dict that merely *looks* like a passing report is refused, and so is a reused report.

In [7]:
fabricated = {"decision": "PASS", "trust_state": "trusted", "gates": {"checkpoint_valid": True}}

tracker = TrustTracker(TrustLabel.RECOVERED)
fresh = store.read_committed()                                              # re-read the persistent state now
B = dict(expected_checkpoint_sha256=attempt.checkpoint_sha256, expected_recovery_id=fresh.recovery_id,
         committed_state_sha256=fresh.state_sha256)
print(tracker.promote_after_verification(fabricated, **B))                         # forged dict
assert tracker.label is TrustLabel.RECOVERED
print(tracker.promote_after_verification(dict(report), **B))                       # a real report flattened to a dict
assert tracker.label is TrustLabel.RECOVERED
print(tracker.apply(TrustLabel.TRUSTED, verified=True, trusted_snapshot=True))  # no back door via apply()
assert tracker.label is TrustLabel.RECOVERED
assert not tracker.promote_after_verification(bad, **B)[0] and tracker.label is TrustLabel.RECOVERED
assert tracker.promote_after_verification(report, **B)[0] and tracker.label is TrustLabel.TRUSTED
print([(h["to"], h["accepted"], h["reason"]) for h in tracker.history])
stale = dict(B, committed_state_sha256="0" * 64)               # the live store no longer matches what was verified
assert TrustTracker(TrustLabel.RECOVERED).promote_after_verification(report, **stale) == (False, "committed_state_mismatch")

(False, 'report_not_issued_by_verify_recovery')
(False, 'report_not_issued_by_verify_recovery')
(False, 'trusted_requires_verification_report')
[('Trusted', False, 'report_not_issued_by_verify_recovery'), ('Trusted', False, 'report_not_issued_by_verify_recovery'), ('Trusted', False, 'trusted_requires_verification_report'), ('Trusted', False, 'verification_not_passed'), ('Trusted', True, 'accepted_by_policy')]


## 6. Contamination classification with a declared uncertainty boundary

In [8]:
observed = [("M1","D1"),("D1","P1"),("P1","A1"),("D2","P2"),("M1","S1")]   # M2->D1 and M2->D2 are missing
result = classify_contamination(observed, ["M2"], declared_missing_edges=[("M2","D1"),("M2","D2")])
print(result)
assert quarantine_set(result) >= {"D1","D2","P1","P2","A1"}
undeclared = classify_contamination(observed, ["M2"])   # nobody told the policy about the missing edges
assert undeclared["potentially_contaminated"] == [] and undeclared["unknown_unverified"] == []
print("Undeclared missing edges are invisible to the policy (blind detection is Lab 01.7).")

{'confirmed': ['M2'], 'potentially_contaminated': [], 'unknown_unverified': ['A1', 'D1', 'D2', 'P1', 'P2']}
Undeclared missing edges are invisible to the policy (blind detection is Lab 01.7).


## 7. Idempotent receiver: a retry after a lost acknowledgement is harmless

In [9]:
receiver = IdempotentReceiver()
assert receiver.store("event-1", {"n": 1}) == "stored"
assert receiver.store("event-1", {"n": 1}) == "duplicate"      # retry
assert len(receiver) == 1
print(receiver.reconcile(["event-1", "event-2"]))

{'missing': ['event-2'], 'unexpected': [], 'intended_has_duplicates': False}


## 8. Limits

Signature validity does not prove signer trustworthiness; the audit log proves consistency, not writer honesty; the manifest's content is an assumption about the real world; the high-water mark is only as safe as its storage. See `docs/limitations.md`.